In [1]:
import tensorflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import os

%load_ext autoreload
%autoreload 2


In [2]:
data_dir = 'data'
df_full = pd.read_csv(os.path.join(data_dir, 'train.csv'))
df_test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
df_full['file'] = df_full['id'].apply(lambda x: f"{x:05d}.mp4")
X = df_full.drop('target', axis=1)
y = df_full['target']

# Split off 70% for training, 20% for temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Submission
df_test['file'] = df_test['id'].apply(lambda x: f"{x:05d}.mp4")
X_submission = df_test

In [3]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [11]:
X_train

,id,time_of_event,time_of_alert,file
382,343,18.826,15.689,00343.mp4
538,330,20.433,19.567,00330.mp4
1493,154,20.333,17.367,00154.mp4
1112,1306,NaN,NaN,01306.mp4
324,577,22.064,20.789,00577.mp4
...,...,...,...,...
1130,426,19.989,19.401,00426.mp4
1294,162,19.983,19.950,00162.mp4
860,2126,NaN,NaN,02126.mp4
1459,309,19.233,18.433,00309.mp4


In [ ]:
def extract_features_dataframe(X, y):
    X_list = []
    for _, row in X.iterrows():
        vid_id = row['id']; fname = row['file']
        video_path = os.path.join(DATA_DIR, "train", fname)
        # Sample frames
        frames = sample_uniform_frames(video_path, num_frames=15)
        if len(frames) == 0:
            # If no frames (corrupt video), skip
            X_list.append(np.zeros((15, 2048), dtype=np.float32))
            continue
        # Preprocess frames for ResNet50
        frames = preprocess_input(frames.astype(np.float32))
        # Extract CNN features for frames
        features = base_cnn.predict(frames, batch_size=15, verbose=0)  # shape (n_frames, 2048)
        X_list.append(features)
    return np.array(X_list, dtype=np.float32)


In [4]:
y_train[:20]

# Get idx of series, print corresponding df_full rows
df_full.iloc[X_train.index]

,id,time_of_event,time_of_alert,target,file
382,343,18.826,15.689,1,00343.mp4
538,330,20.433,19.567,1,00330.mp4
1493,154,20.333,17.367,1,00154.mp4
1112,1306,NaN,NaN,0,01306.mp4
324,577,22.064,20.789,1,00577.mp4
...,...,...,...,...,...
1130,426,19.989,19.401,1,00426.mp4
1294,162,19.983,19.950,1,00162.mp4
860,2126,NaN,NaN,0,02126.mp4
1459,309,19.233,18.433,1,00309.mp4


In [ ]:
import cv2
from utils import sample_uniform_frames
import numpy as np
from tqdm.notebook import tqdm

X_train_frames = []
for video in tqdm(X_train['file'][:10]):
    frames = sample_uniform_frames(os.path.join(data_dir, 'train', video), num_frames=32)
    X_train_frames.append(frames)

X_train_frames = np.array(X_train_frames)
X_train_frames.shape



  0%|          | 0/10 [00:00<?, ?it/s]

(10, 32, 224, 224, 3)